In [ ]:
import duckdb
import pandas as pd

In [ ]:
def load(slice_name):
    return duckdb.connect().sql(f"""
        SELECT * REPLACE (
            coalesce(postcode_clean, '<NONE>') AS postcode_clean,
            coalesce(street_clean,   '<NONE>') AS street_clean
        )
        FROM 'data/raw/{slice_name}_linkage_input.parquet'
    """).df()

def street_key_expr(col: str = "street_clean") -> str:
    """
    Normalise a cleaned street name for grouping: collapses spacing
    variants and a trailing plural/possessive 's'. Does not attempt
    to fix missing/extra words - only exact respellings of the same
    tokens.
    """
    return f"""
        regexp_replace(
            regexp_replace({col}, '[^A-Z0-9]', '', 'g'),  -- drop spaces/apostrophes
            'S$', ''                                       -- drop one trailing S
        )
    """

def ex_key_match(slice_name):
    df = load(slice_name)
    street_key = street_key_expr("street_clean")
    return duckdb.connect().sql(f"""
        SELECT *,
               dense_rank() OVER (
                   ORDER BY postcode_clean, paon_clean, unit_key, street_key
               ) AS property_id
        FROM (
            SELECT *, {street_key} AS street_key
            FROM df
        )
    """).df()

In [ ]:
res_bris = ex_key_match("bristol")
res_pow = ex_key_match("powys")

display(res_bris[["property_id", "postcode_clean", "paon_clean", "street_clean", "street_key"]].head())
display(res_pow[["property_id", "postcode_clean", "paon_clean", "street_clean", "street_key"]].head())

In [ ]:
count_bris = res_bris.groupby("property_id").size()
count_pow  = res_pow.groupby("property_id").size()

summary = pd.DataFrame({
    "bristol": {
        "n_properties": count_bris.size,
        "n_sales": count_bris.sum(),
        "biggest_group": count_bris.max(),
    },
    "powys": {
        "n_properties": count_pow.size,
        "n_sales": count_pow.sum(),
        "biggest_group": count_pow.max(),
    },
})
summary

In [ ]:
con = duckdb.connect()
con.sql("""
    SELECT postcode_clean, paon_clean, unit_key, street_key,
           count(DISTINCT street_clean) AS n_variants,
           array_agg(DISTINCT street_clean) AS variants
    FROM res_bris
    GROUP BY postcode_clean, paon_clean, unit_key, street_key
    HAVING count(DISTINCT street_clean) > 1
""").show(max_rows=20)

con.sql("""
    SELECT postcode_clean, paon_clean, unit_key, street_key,
           count(DISTINCT street_clean) AS n_variants,
           array_agg(DISTINCT street_clean) AS variants
    FROM res_pow
    GROUP BY postcode_clean, paon_clean, unit_key, street_key
    HAVING count(DISTINCT street_clean) > 1
""").show(max_rows=60)